# Imports

In [18]:
from nb_utils import set_root
PROJECT_DIR = set_root(2)

In [19]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import clear_output, display


# Parameters

In [20]:
path_data = PROJECT_DIR / "data"
path_primary = path_data / "03_primary"

file_path_data = path_primary / "horm_tracker.parquet"
tracker_columns = ["tracker_id",	"class_id",	"x_min",	"y_min",	"x_max",	"y_max",	"x_center",	"y_center"]

# Read

In [ ]:
data = pd.read_parquet(file_path_data)
data.head()

In [22]:
data["ID"] = data["ID"].astype(int)
data = data.set_index("ID")

# Example: ID = 35

In [52]:
data_filter = data[data.index == 14].copy()

## Viz

In [53]:
unique_tracker_ids = data_filter["tracker_id"].unique()

fig, ax = plt.subplots(1, 1, figsize=(16, 8))

for id_tracker in unique_tracker_ids:
    data_id = data_filter[data_filter["tracker_id"] == id_tracker]
    ax.plot(data_id["x_center"], data_id["y_center"], label=f"Tracker ID: {id_tracker}")

# Configurações do gráfico
ax.set_title("Trajetórias dos Trackers")
ax.set_xlabel("X Center")
ax.set_ylabel("Y Center")
ax.spines[["top", "right"]].set_visible(False)
#ax.legend()  # Adiciona a legenda
plt.show()

In [ ]:
# Widget dropdown para selecionar a classe
classe_dropdown = widgets.Dropdown(
    options=data_filter["tracker_id"].unique(),
    description="Classe:"
)

# Função para gerar o scatter plot com base na classe selecionada
def plot_scatter(classe):
    clear_output(wait=True)  # Limpa a saída antes de gerar um novo gráfico
    display(classe_dropdown)  # Redesenha o dropdown
    filtered_df = data_filter[data_filter["tracker_id"] == classe]
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=filtered_df["x_center"],
        y=filtered_df["y_center"],
        mode="lines+markers",
        marker=dict(size=10),
        name=f"Classe {classe}"
    ))
    fig.update_layout(
        title=f"Scatter Plot para Classe {classe}",
        xaxis_title="Eixo X",
        yaxis_title="Eixo Y",
        showlegend=True,
        height=600
    )
    fig.show()

# Exibe o dropdown e inicializa a interação
display(classe_dropdown)
classe_dropdown.observe(lambda change: plot_scatter(change["new"]), names="value")

# Comparison

In [57]:
unique_ids = [82, 14, 29, 52] #[35, 19, 38, 24]  #[47, 35, 36, 82]#list(data.index.unique())[:2]

In [ ]:
fig, ax = plt.subplots(len(unique_ids) // 2, 2, figsize=(16, 12))
if isinstance(ax, np.ndarray):
    ax = ax.flatten()
for id_instance, _ax in zip(unique_ids, ax):
    data_filter = data[data.index == id_instance].copy()
    unique_tracker_ids = data_filter["tracker_id"].unique()
    for id_tracker in unique_tracker_ids:
        data_id = data_filter[data_filter["tracker_id"] == id_tracker]
        _ax.plot(data_id["x_center"], data_id["y_center"], label=f"Tracker ID: {id_tracker}")

    # Configurações do gráfico
    _ax.set_title(f"ID: {id_instance}")
    _ax.set_xlabel("X Center")
    _ax.set_ylabel("Y Center")
    _ax.spines[["top", "right"]].set_visible(False)
    #_ax.legend()  # Adiciona a legenda
    #plt.show()

In [ ]:
data.loc[unique_ids, :].drop(tracker_columns, axis=1).drop_duplicates()[["Total sperm count", "Seminal AMH"]]

In [ ]:
data[tracker_columns].reset_index().drop(["tracker_id", "class_id"], axis=1).groupby("ID").var().idxmin()

In [ ]:
data[tracker_columns].reset_index().drop(["tracker_id", "class_id"], axis=1).groupby("ID").var().idxmax()

In [ ]:
data.drop(tracker_columns, axis=1).loc[unique_ids, :].drop_duplicates()[["Total sperm count", "Seminal AMH", "BMI", "Age"]]

In [ ]:
data.reset_index()

# Teste graphs

In [34]:
data_temp = data.reset_index().copy()

In [ ]:

import gravis as gv
import networkx as nx
# Filter DataFrame
filtered_df = data_temp[(data_temp.ID == 14) & (data_temp.tracker_id == 48)].iloc[:, :9].reset_index(drop=True).copy()

# Create Graph
G = nx.DiGraph(directed=True)

# Add Nodes
for idx, row in filtered_df.iterrows():
    G.add_node(idx, pos=(row['x_center'], row['y_center']))

# Add Edges
for i in range(len(filtered_df) - 1):
    G.add_edge(i, i + 1)

# Get positions
pos = nx.get_node_attributes(G, 'pos')

gv.d3(
    G,
    use_node_size_normalization=True,
    node_size_normalization_max=30,
    use_edge_size_normalization=True,
    edge_size_data_source='weight',
    
    # edge_curvature=0.3,
    zoom_factor=0.6
)

In [32]:
# import matplotlib.pyplot as plt
#  # Define a posição dos nós
# k = 4
# i_graph = graphs[k]
# pos = nx.spring_layout(i_graph)

# # Desenha o grafo
# plt.figure(figsize=(10, 8))
# nx.draw(i_graph, pos, with_labels=True, node_color='skyblue', font_size=5, font_weight='bold')

# # Desenha as arestas com largura proporcional ao peso
# edges = i_graph.edges(data=True)
# nx.draw_networkx_edges(i_graph, pos, edgelist=edges, width=[d['weight_inv'] for (u, v, d) in edges])

# plt.title(f'Grafo para o Livro {k}')
# plt.show()